# Day 04：容器数据结构实战 —— 任务队列 (`list`)、不可变配置 (`tuple`) 与标签去重 (`set`)

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》实战第 4 天！我是你的 Tech Lead 张工。前三天的代码里，我们一直在处理单个字符串或单个数字；但真实的 Agent 每天都要面对**成百上千条待办任务、多组大模型配置白名单、以及多路 RAG 检索回来的重复标签**。今天我们用企业级任务队列 `task_queue.py` 作为实战载体，彻底掌握 Python 三大基础容器：`list`、`tuple` 和 `set`。

## 1. 第一步：使用不可变元组 `tuple` 锁定系统模型白名单
为什么系统受支持的大模型列表要用 `tuple`（圆括号 `(...)`）而不是 `list`（方括号 `[...]`）？
- **工程安全性**：`tuple` 一旦创建，**不允许原地增删改**，能防止团队其他模块在运行期不小心 `.append()` 或修改核心配置。
- **元组解包**：可以一行代码把配置项拆解到多个变量中。

In [1]:
# 定义系统全局受支持的大模型白名单（使用 tuple 保证运行期不可篡改）
SUPPORTED_MODELS: tuple[str, ...] = (
    "deepseek-chat",
    "deepseek-reasoner",
    "qwen-max",
    "qwen-plus",
    "glm-4-plus",
)

print("白名单类型:", type(SUPPORTED_MODELS))
print("模型数量:", len(SUPPORTED_MODELS))
print("deepseek-chat 是否受支持:", "deepseek-chat" in SUPPORTED_MODELS)

# 演示元组解包（Tuple Unpacking）：提取首选模型、备选模型与其他模型
primary_model, reasoning_model, *backup_models = SUPPORTED_MODELS
print(f"主模型: {primary_model} | 推理模型: {reasoning_model} | 备选列表: {backup_models}")

白名单类型: <class 'tuple'>
模型数量: 5
deepseek-chat 是否受支持: True
主模型: deepseek-chat | 推理模型: deepseek-reasoner | 备选列表: ['qwen-max', 'qwen-plus', 'glm-4-plus']


## 2. 第二步：使用可变列表 `list` 管理 Agent 任务队列（增删改查、列表推导式与切片）
在 `TodoManager` 中，任务队列是动态变化的，因此必须使用 `list`：
- `append()`：向队列尾部追加新任务
- **列表推导式**：`[t for t in tasks if not t["done"]]` 一行筛选未完成任务
- **切片（Slicing）**：`tasks[:k]` 快速提取前 K 个优先处理的任务

In [2]:
# 初始化一个空的任务队列列表
task_queue: list[dict] = []

# 1. 追加任务（list.append）
task_queue.append({"title": "配置企业知识库索引", "priority": "high", "done": False})
task_queue.append({"title": "清洗历史工单 Prompt", "priority": "normal", "done": False})
task_queue.append({"title": "生成周报摘要", "priority": "low", "done": False})
task_queue.append({"title": "归档测试日志", "priority": "low", "done": True})

# 2. 标记第二个任务已完成
task_queue[1]["done"] = True

# 3. 使用列表推导式筛选所有尚未完成的待办任务
pending_tasks = [t for t in task_queue if not t["done"]]
print("待办任务（列表推导式）:", [t["title"] for t in pending_tasks])

# 4. 使用列表切片获取前 2 个任务（Top-K）
top_2 = task_queue[:2]
print("Top-2 任务（切片）:", [t["title"] for t in top_2])

# 5. 使用列表过滤原地移除指定标题的任务
task_queue[:] = [t for t in task_queue if t["title"] != "归档测试日志"]
print("移除归档任务后剩余数量:", len(task_queue))

待办任务（列表推导式）: ['配置企业知识库索引', '生成周报摘要']
Top-2 任务（切片）: ['配置企业知识库索引', '清洗历史工单 Prompt']
移除归档任务后剩余数量: 3


## 3. 第三步：使用集合 `set` 进行 RAG 知识库标签去重与集合运算
在后续第三阶段做企业 RAG 知识库检索时，向量检索和关键词检索（BM25）经常会召回重复标签。`set` 天然具备**元素唯一性**和**数学集合运算**：
- 并集 `|`：合并两路标签并自动去重
- 交集 `&`：找出两路检索都命中的高置信度标签
- 差集 `-`：找出仅由某一路检索发现的增量标签

In [3]:
def merge_unique_tags(tags_a: list[str], tags_b: list[str]) -> set[str]:
    """使用集合推导式清洗空白并执行并集去重。"""
    set_a = {tag.strip() for tag in tags_a if tag.strip()}
    set_b = {tag.strip() for tag in tags_b if tag.strip()}
    return set_a | set_b

vector_tags = ["合同", "报销", "差旅", "报销", " 财务制度 "]
bm25_tags = ["差旅", "审批流", "财务制度", "HR", ""]

# 1. 并集去重
merged = merge_unique_tags(vector_tags, bm25_tags)
print("并集去重结果:", sorted(merged))

# 2. 交集与差集演示
set_v = {t.strip() for t in vector_tags if t.strip()}
set_b = {t.strip() for t in bm25_tags if t.strip()}
print("交集（共同命中）:", sorted(set_v & set_b))
print("差集（仅向量命中）:", sorted(set_v - set_b))

并集去重结果: ['HR', '合同', '审批流', '差旅', '报销', '财务制度']
交集（共同命中）: ['差旅', '财务制度']
差集（仅向量命中）: ['合同', '报销']


## 4. 第四步：实战串联 `TodoManager`（对应 `project_iterations/day_04`）
将上述三大容器整合到工程级 `TodoManager` 中，验证完整的任务入队、完成、筛选与 Top-K 提取。

In [4]:
class DemoTodoManager:
    """Day 04 任务队列管理器演示版"""
    def __init__(self) -> None:
        self._tasks: list[dict] = []

    def add_task(self, title: str, priority: str = "normal") -> None:
        self._tasks.append({"title": title.strip(), "priority": priority, "done": False})

    def complete_task(self, title: str) -> bool:
        for t in self._tasks:
            if t["title"] == title.strip():
                t["done"] = True
                return True
        return False

    def get_pending_tasks(self) -> list[dict]:
        # 列表推导式过滤未完成任务
        return [t for t in self._tasks if not t["done"]]

    def top_k_tasks(self, k: int) -> list[dict]:
        # 切片提取前 k 个任务
        return self._tasks[:k] if k > 0 else []

manager = DemoTodoManager()
manager.add_task("接入 DeepSeek 模型", priority="high")
manager.add_task("编写 Day 04 单元测试", priority="high")
manager.add_task("整理 RAG 标签库", priority="normal")
manager.complete_task("接入 DeepSeek 模型")

print("当前待办任务:", [t["title"] for t in manager.get_pending_tasks()])
print("前 2 项任务:", [t["title"] for t in manager.top_k_tasks(2)])

当前待办任务: ['编写 Day 04 单元测试', '整理 RAG 标签库']
前 2 项任务: ['接入 DeepSeek 模型', '编写 Day 04 单元测试']
